In [2]:
# pip install rapidfuzz

from rapidfuzz import fuzz, process

# --- 1. Basic similarity scores (0-100) ---
print(fuzz.ratio("hello world", "hello wrld"))          # simple similarity
print(fuzz.partial_ratio("hello", "say hello there"))     # best matching substring
print(fuzz.token_sort_ratio("new york mets", "mets new york"))  # ignores word order
print(fuzz.token_set_ratio("new york mets vs", "new york mets"))  # ignores duplicates/extra words

# --- 2. Find the best match in a list ---
choices = ["New York Mets", "New York Yankees", "Boston Red Sox", "Chicago Cubs"]

best_match = process.extractOne("ny mets", choices)
print(best_match)  # ('New York Mets', 90.0, 0) -> (match, score, index)

# --- 3. Get top N matches ---
top_matches = process.extract("new yrok", choices, limit=2)
print(top_matches)

# --- 4. Fuzzy matching with a score cutoff (filter out weak matches) ---
matches = process.extract("boston sox", choices, score_cutoff=60)
print(matches)

# --- 5. Deduplicating a list of similar strings ---
# NOTE: process.dedupe does not exist in current RapidFuzz — build it manually
def dedupe(strings, threshold=85):
    """Keep only one representative per group of near-duplicate strings."""
    kept = []
    for s in strings:
        match = process.extractOne(s, kept, scorer=fuzz.ratio, score_cutoff=threshold)
        if match is None:
            kept.append(s)
    return kept

data = ["apple inc", "apple incorporated", "Apple Inc.", "banana co", "Banana Co."]
deduped = dedupe(data, threshold=85)
print(deduped)

# --- 6. Matching against a dictionary (keys as choices, get back the key) ---
db = {"nyc": "New York City", "la": "Los Angeles", "sf": "San Francisco"}
match, score, key = process.extractOne("san fransisco", db)
print(match, score, key)

# --- 7. Full similarity matrix (all-pairs comparison, useful for clustering) ---
import numpy as np

matrix = process.cdist(data, data, scorer=fuzz.ratio)
print(np.round(matrix, 1))

95.23809523809523
100.0
100.0
100.0
('New York Mets', 60.00000000000001, 0)
[('New York Mets', 57.0, 0), ('New York Yankees', 56.25, 1)]
[('Boston Red Sox', 66.66666666666667, 2)]
['apple inc', 'apple incorporated', 'Apple Inc.', 'banana co', 'Banana Co.']
San Francisco 76.92307692307692 sf
[[100.   66.7  73.7  33.3  21.1]
 [ 66.7 100.   50.   29.6  21.4]
 [ 73.7  50.  100.   21.1  20. ]
 [ 33.3  29.6  21.1 100.   73.7]
 [ 21.1  21.4  20.   73.7 100. ]]


In [3]:
from rapidfuzz import fuzz, process

# ---------------------------------------------------------------------
# 1. Comparing Two Strings (Scoring from 0 to 100)
# ---------------------------------------------------------------------
string1 = "Rowing Machine"
string2 = "Rowing Machin"

# Basic Ratio: Compares the exact sequence
ratio = fuzz.ratio(string1, string2)
print(f"Basic Ratio: {ratio:.1f}%") 
# Output: ~96.3%

# Partial Ratio: Great if one string is a substring of the other
string3 = "The Rowing Machine in the gym"
partial_ratio = fuzz.partial_ratio(string1, string3)
print(f"Partial Ratio: {partial_ratio:.1f}%") 
# Output: 100.0%

# Token Sort Ratio: Ignores word order completely
string4 = "Machine Rowing"
token_sort = fuzz.token_sort_ratio(string1, string4)
print(f"Token Sort Ratio: {token_sort:.1f}%") 
# Output: 100.0%


# ---------------------------------------------------------------------
# 2. Searching a List (Process Matching)
# ---------------------------------------------------------------------
choices = ["Apple iPhone 15", "Samsung Galaxy S24", "Google Pixel 8", "Apple iPad Pro"]
query = "iphone 15"

print("\n--- Searching the List ---")

# Extract the single best match
# Returns a tuple: (match, score, index)
best_match = process.extractOne(query, choices)
print(f"Best Match: '{best_match[0]}' with a score of {best_match[1]:.1f}% at index {best_match[2]}")

# Extract multiple top matches (e.g., top 2)
top_matches = process.extract(query, choices, limit=2)
print("\nTop 2 Matches:")
for match, score, index in top_matches:
    print(f"- {match} (Score: {score:.1f}%)")

Basic Ratio: 96.3%
Partial Ratio: 100.0%
Token Sort Ratio: 100.0%

--- Searching the List ---
Best Match: 'Apple iPhone 15' with a score of 85.5% at index 0

Top 2 Matches:
- Apple iPhone 15 (Score: 85.5%)
- Google Pixel 8 (Score: 36.0%)
